# 🛡️ Phishing URL Detection: Training Pipeline
This notebook is the official pipeline for loading raw URL data, extracting heuristic features, and training advanced Machine Learning models (including XGBoost and Random Forest).

**Note**: The heavy lifting for parallel feature extraction is safely modularized in `feature_extraction.py`.


In [ ]:
import os
import pandas as pd
import numpy as np
import joblib
from sklearn.model_selection import train_test_split, RandomizedSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from xgboost import XGBClassifier

# Import custom modules
from feature_extraction import extract_features_parallel, get_feature_names
from evaluate import (
    evaluate_model,
    plot_confusion_matrix,
    plot_model_comparison,
    plot_feature_importance,
    plot_class_distribution,
)

# For inline notebook plotting
%matplotlib inline

In [ ]:
def prepare_data(raw_csv_path="urldata.csv", extracted_csv_path="urldata_extracted.csv"):
    needs_extraction = True

    if os.path.exists(extracted_csv_path):
        try:
            extracted_count = pd.read_csv(extracted_csv_path, usecols=["target"]).shape[0]
            raw_count = pd.read_csv(raw_csv_path, usecols=["url"]).shape[0]
        except Exception as e:
            print(f"Invalid cached file: {e}")
            os.remove(extracted_csv_path)
        else:
            if extracted_count >= int(raw_count * 0.99):
                print(f"Loading pre-extracted features from '{extracted_csv_path}' ({extracted_count} rows)...")
                df = pd.read_csv(extracted_csv_path)
                needs_extraction = False
            else:
                print("Stale cache detected. Re-extracting...")
                os.remove(extracted_csv_path)

    if needs_extraction:
        print(f"Reading raw data from '{raw_csv_path}'...")
        df_raw = pd.read_csv(raw_csv_path)
        
        target_col = "result" if "result" in df_raw.columns else "label"
        targets = np.where(df_raw[target_col].str.lower() == "benign", 0, 1) if df_raw[target_col].dtype == object else df_raw[target_col].values
        urls = df_raw["url"].values

        print(f"Extracting features for {len(urls):,} URLs...")
        features_list = extract_features_parallel(urls)

        df = pd.DataFrame(features_list)
        df["target"] = targets
        df["url"] = urls
        df.to_csv(extracted_csv_path, index=False)

    return df[get_feature_names()], df["target"]

# Run Data Preparation
X, y = prepare_data()
feature_names = X.columns.tolist()
print(f"\nTotal usable samples: {len(X):,}")

In [ ]:
# Visualize the class balance
plot_class_distribution(y)

In [ ]:
# Train / Test split (80/20)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print(f"Train samples: {len(X_train):,} | Test samples: {len(X_test):,}")

# Feature scaling
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [ ]:
# Define Models
models = {
    "Decision Tree": DecisionTreeClassifier(random_state=42),
    "Random Forest": RandomForestClassifier(n_estimators=300, n_jobs=-1, random_state=42),
    "Logistic Regression": LogisticRegression(max_iter=1000, random_state=42, n_jobs=-1),
    "XGBoost": XGBClassifier(
        n_estimators=200, max_depth=6, learning_rate=0.1, subsample=0.8,
        colsample_bytree=0.8, eval_metric="logloss", tree_method="hist",
        random_state=42, n_jobs=-1
    ),
}

results = {}
best_model_name = None
best_f1 = 0.0
best_model = None

# Train & Evaluate
for name, model in models.items():
    print(f"\n─── Training : {name} ───")
    model.fit(X_train_scaled, y_train)
    y_pred = model.predict(X_test_scaled)
    
    metrics = evaluate_model(name, y_test, y_pred)
    results[name] = metrics
    
    plot_confusion_matrix(name, y_test, y_pred)
    
    if metrics["F1"] > best_f1:
        best_f1 = metrics["F1"]
        best_model_name = name
        best_model = model

    if name in ("Random Forest", "XGBoost"):
        plot_feature_importance(model, feature_names, model_name=name)

In [ ]:
# Model Comparison
plot_model_comparison(results)

In [ ]:
# Hyperparameter Tuning on Best Model
print(f"Best initial model: {best_model_name} | F1 = {best_f1:.4f}")

param_grid = {}
if best_model_name == "XGBoost":
    param_grid = {
        "n_estimators": [200, 300, 500],
        "max_depth": [4, 6, 8],
        "learning_rate": [0.05, 0.1, 0.2]
    }
elif best_model_name == "Random Forest":
    param_grid = {
        "n_estimators": [300, 500, 700],
        "max_depth": [None, 20, 30],
        "min_samples_split": [2, 5, 10]
    }

if param_grid:
    # Notice n_iter=5 for faster completion!
    search = RandomizedSearchCV(
        best_model, param_grid, n_iter=5, cv=3, scoring="f1", n_jobs=-1, random_state=42, verbose=1
    )
    search.fit(X_train_scaled, y_train)
    
    print(f"\nBest params : {search.best_params_}")
    tuned_model = search.best_estimator_
    
    y_pred_tuned = tuned_model.predict(X_test_scaled)
    tuned_metrics = evaluate_model(f"{best_model_name} (Tuned)", y_test, y_pred_tuned)
    
    if tuned_metrics["F1"] >= best_f1:
        print(f"\nTuned model F1 : {tuned_metrics['F1']:.4f} (improved from {best_f1:.4f})")
        best_model = tuned_model
        best_f1 = tuned_metrics["F1"]

In [ ]:
# Save the final production model and scaler
os.makedirs("models", exist_ok=True)

joblib.dump(best_model, "models/best_model.pkl")
joblib.dump(scaler, "models/scaler.pkl")

print(f"DONE — Saved Best Model ({best_model_name}) and Scaler to models/ directory.")